In [42]:
# 
# Dowloading data from CDS requires:
# 1. setting up account on CDS cds.copernicus-climate.eu
# 2. installation of python cdsapi library
# 3. creation of .ecmwfapi file with credentials from the CDS account

import cdsapi
import xarray as xr
import datetime, glob, sys, os
import numpy as np
import pandas as pd

In [43]:
def download_reanalysis(outputfile,year, month):

    #simiple function to download monthly rainfall forecast by ECMWF SEAS5.1 model
    
    # forecast system, variable, definition of region in hardcoded
    
    #this defines dataset
    dataset = "seasonal-monthly-single-levels"

    #this defines dataset
    dataset = "reanalysis-era5-single-levels-monthly-means"

    #this defines dataset dictionary
    request = {
        "product_type": ["monthly_averaged_reanalysis"],
        "variable": ["total_precipitation"],
        "year": [str(year)],
        "month": [str(month).zfill(2)],
        "time": ["00:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [39, 59, 29, 75]
    }

    print(request)
    
    client = cdsapi.Client()

    try:
        #requesting data
        client.retrieve(dataset, request, outputfile)
    except Exception as e:
        print(e)
        print("Something went wrong. Cannot download data from CDS".format(outputfile))

    #checking if file can be opened
    try:
        ds=xr.open_dataset(outputfile)
        print(ds.dims)
        ds.close()
    except Exception as e:
        print(e)
        print("Something went wrong. Downloaded file cannot be read \n{}".format(outputfile))

    print(f"Successfully downloaded {outputfile}")


In [44]:
# to issue forecast, historical data have to be available for each month till the current month. 
# All files present in the output directory will be read and concatenated 
# and the most recent data will be identified. Download function could then be called to 
# update the stash of data till current date

#data directory
datadir="./data/ERA5_rainfall/"

files=glob.glob("{}/tp_mon_ERA5_afghanistan_*.nc".format(datadir))

if len(files)==0:
    print("there appears to be no data files in the {} directory. That should not be the case. Please check if data directory is correct.".format(datadir))
    sys.exit()


dates=[pd.to_datetime(x.split("_")[-1][0:6], format="%Y%m") for x in np.sort(files)]
dates=pd.to_datetime(dates)



#the most recent date available locally
firstdatadate=dates[0]
lastdatadate=dates[-1]

print(f"last data available on the system: {lastdatadate.strftime('%b %Y')}")

today=pd.to_datetime(datetime.datetime.now()).normalize()
currentday=today.day
currentdate=today.replace(day=1)

if currentday>5:
    lastreanalysisdate=currentdate-pd.offsets.MonthBegin(1)
else:
    lastreanalysisdate=currentdate-pd.offsets.MonthBegin(2)
print(f"last reanalysis date available on CDS: {lastreanalysisdate}")

for date in pd.date_range(firstdatadate, lastreanalysisdate, freq="MS"):
    
    year=date.year
    month=date.month

    #output file
    outputfile="{}/tp_mon_ERA5_afghanistan_{}{}.nc".format(datadir,year,str(month).zfill(2))
    if not os.path.exists(outputfile):
        print(f"Downloading {outputfile}")
        
        download_reanalysis(outputfile,year,month)
        
    elif date==lastreanalysisdate:
        print("Data for the most recent month are already downloaded. Exiting...")


last data available on the system: Feb 2026
last reanalysis date available on CDS: 2026-03-01 00:00:00
{'product_type': ['monthly_averaged_reanalysis'], 'variable': ['total_precipitation'], 'year': ['2026'], 'month': ['03'], 'time': ['00:00'], 'data_format': 'netcdf', 'download_format': 'unarchived', 'area': [39, 59, 29, 75]}


2026-04-26 11:17:43,030 WARNING [2026-04-26T09:17:42.880388] You are using a deprecated API endpoint. If you are using cdsapi, please upgrade to the latest version.
2026-04-26 11:17:43,032 INFO Request ID is cb3e428b-0212-42e2-9c27-c51a8c13de97
2026-04-26 11:17:43,271 INFO status has been updated to accepted
2026-04-26 11:17:56,363 INFO status has been updated to running
2026-04-26 11:18:04,182 INFO status has been updated to successful


dbfc02195a9e98fafec287dbeef247f.nc:   0%|          | 0.00/29.7k [00:00<?, ?B/s]

FrozenMappingWarningOnValuesAccess({'valid_time': 1, 'latitude': 41, 'longitude': 65})
Successfully downloaded ./data/ERA5_rainfall//tp_mon_ERA5_afghanistan_202603.nc
